### Deps

In [ ]:
import openai
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery


In [ ]:
### Load Environment
from dotenv import load_dotenv #import 
import os 

load_dotenv("../../.env")

### Retrieval Pipeline

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
query = "can I get a tablet?"

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=embedding,
                using='text-embedding-3-small', #name of the vector in the collection
                limit=20
            ), 
            Prefetch(
                query=Document(text=query, model='qdrant/bm25'),
                using='bm25', #name of the vector in the collection
                limit=20
            )
        ],
        # can set specific weights on how much importance we put on dense embeddings and bm25 lexical retrievals
        # in this case dense vector retrival is weighted as 3 times more important than bm25
        # if user queries are more keyword-based, you might want to increase the weight of bm25
        # if more contextual then you might want to increase the weight of the dense vector retrieval
        # weights can be added as params to the retrieve data function
        query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])), # how you fuse the results....
        limit=k # once fused we are returning top k results
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


In [ ]:
results = retrieve_data(query, k=20)

print(results)

### Reranking

In [ ]:
import cohere

cohere_client = cohere.ClientV2()

In [ ]:
to_rerank = results["retrieved_context"]




In [ ]:
# the list gets shuffled to the ranking
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [ ]:
response

In [ ]:
reranked_results = [to_rerank[result.index] for result in response.results]
    
    

In [ ]:
reranked_results